# Phase 03D: Teacher A Seed 2026 Full Training

Teacher A:

`CLIP ViT-B/32 + visual adapter 768→256 + shared caption decoder`

Notebook này:

- verify Phase 03B và Phase 03C;
- dựng Train/Validation DataLoader;
- tải offline CLIP và initial adapter/decoder;
- dựng COCO metric backend offline;
- smoke test Teacher-forced Validation và Beam-3;
- chạy discarded production pilot;
- huấn luyện full Teacher A seed 2026;
- lưu atomic last/best checkpoints;
- chọn best checkpoint chỉ bằng Validation CIDEr-D;
- đánh giá quality gate so với S1 seed 2026;
- không mở Held-out Test.

## Kaggle inputs

1. Phase 1 Formal Output.
2. Phase 03B Teacher A Preflight.
3. Phase 03C Teacher A Training Readiness.
4. MS COCO 2014 images.
5. Offline `openai/clip-vit-base-patch32`.
6. Offline wheel `pycocoevalcap-1.2-py3-none-any.whl`.

## Cấu hình

- GPU T4.
- Internet Off.
- Chạy tuần tự từng cell.
- Cell 7 là full training và có thể chạy nhiều giờ.


## Confirmation-seed protocol

- Run seed: `2026`.
- Shared initialization checkpoint: seed `42`, giống protocol ba seed của
  Phase 2 để giữ cùng điểm xuất phát.
- Seed `2026` điều khiển sampler, stateless augmentation, dropout và RNG
  trong huấn luyện.
- Quality gate được so với S1 cùng seed `2026`.


## Cell 1 — Verify Phase 03B/03C và environment

### Công việc

- khóa seed 2026 và deterministic backend;
- verify Phase 03C READY marker và manifest;
- verify Phase 03B initial checkpoint;
- tìm offline CLIP và offline metric wheel;
- chưa đọc ảnh hoặc caption.

### Thời gian dự kiến

15–120 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Phase 03C                 : VERIFIED
Teacher A training engine : READY
Offline CLIP              : FOUND
Metric wheel              : FOUND
Progress                  : 1 / 8 cells passed
```


In [ ]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 2026
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = (
    Path("/kaggle/working")
    / ""
    / "phase3_teacher_a_seed2026"
)
CONFIG_DIR = OUTPUT_ROOT / "configs"
AUDIT_DIR = OUTPUT_ROOT / "audits"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LOG_DIR = OUTPUT_ROOT / "logs"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    CONFIG_DIR,
    AUDIT_DIR,
    CHECKPOINT_DIR,
    LOG_DIR,
    VALIDATION_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with open(path, "r", encoding="utf-8") as source:
        return json.load(source)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        while True:
            chunk = source.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    candidate = (root / relative_path).resolve()
    assert candidate.is_relative_to(root)
    return candidate


def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


assert KAGGLE_INPUT_ROOT.is_dir()
assert torch.cuda.is_available(), "Hãy bật GPU T4."
set_deterministic(SEED)
device = torch.device("cuda")

phase3c_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE3_TEACHER_A_FULL_TRAINING_READY.json"
    )
)
assert len(phase3c_matches) == 1, (
    f"Cần đúng một Phase 03C, thấy {len(phase3c_matches)}."
)
PHASE3C_READY_PATH = phase3c_matches[0].resolve()
PHASE3C_ROOT = PHASE3C_READY_PATH.parent
PHASE3C_READY = read_json(PHASE3C_READY_PATH)
assert PHASE3C_READY["status"] == "ready"
assert PHASE3C_READY["teacher_id"] == "A"
assert all(
    value == "passed" or value is True
    for value in PHASE3C_READY["gates"].values()
)
assert PHASE3C_READY["full_training_started"] is False
assert PHASE3C_READY["test_set_status"] == "sealed"
assert PHASE3C_READY["test_set_used"] is False

manifest_record = PHASE3C_READY["manifests"]["csv"]
manifest_path = resolve_inside(
    PHASE3C_ROOT, manifest_record["relative_path"]
)
assert sha256_file(manifest_path) == manifest_record["sha256"]
with open(
    manifest_path,
    "r",
    encoding="utf-8",
    newline="",
) as source:
    phase3c_rows = list(csv.DictReader(source))
for row in phase3c_rows:
    artifact = resolve_inside(PHASE3C_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

phase3b_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE3_TEACHER_A_PREFLIGHT_READY.json"
    )
)
assert len(phase3b_matches) == 1, (
    "Thiếu hoặc gắn trùng Phase 03B Teacher A CLIP Preflight. "
    "Hãy chọn Add Input > Your Work > Notebooks và gắn output "
    "của notebook 03B_Teacher_A_CLIP_ViT_B32_Preflight. "
    f"Số READY marker tìm thấy: {len(phase3b_matches)}. "
    f"Matches: {[str(path) for path in phase3b_matches]}"
)
PHASE3B_ROOT = phase3b_matches[0].resolve().parent
INITIAL_LIGHT_CHECKPOINT = (
    PHASE3B_ROOT
    / "checkpoints"
    / "teacher_a_adapter_decoder_initial_seed42.pt"
)
TRAINING_CONTRACT_PATH = (
    PHASE3B_ROOT
    / "configs"
    / "teacher_a_training_validation_contract.json"
)
assert INITIAL_LIGHT_CHECKPOINT.is_file()
assert TRAINING_CONTRACT_PATH.is_file()
TRAINING_CONTRACT = read_json(TRAINING_CONTRACT_PATH)
assert TRAINING_CONTRACT["status"] == "locked"
BASE_CONTRACT_DEVELOPMENT_SEED = int(
    TRAINING_CONTRACT["reproducibility"]["development_seed"]
)
assert BASE_CONTRACT_DEVELOPMENT_SEED == 42
MATCHED_STUDENT_CIDER_D = 0.9784506928348418
QUALITY_THRESHOLD_OVERRIDE = 0.9794506928348418
RUN_SEED_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_a_seed2026_run_seed_contract.json"
)
write_json(
    RUN_SEED_CONTRACT_PATH,
    {
        "schema_version": "phase3_teacher_a_confirmation_seed_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "teacher_id": "A",
        "run_seed": SEED,
        "base_contract_development_seed": (
            BASE_CONTRACT_DEVELOPMENT_SEED
        ),
        "shared_initialization_seed": 42,
        "shared_initialization_checkpoint": (
            "teacher_a_adapter_decoder_initial_seed42.pt"
        ),
        "seed_controls": [
            "image_balanced_sampler",
            "stateless_augmentation",
            "dropout",
            "python_numpy_torch_cuda_rng",
        ],
        "matched_student": "S1 MobileNetV3-Large",
        "matched_student_seed": SEED,
        "matched_student_validation_cider_d": (
            MATCHED_STUDENT_CIDER_D
        ),
        "minimum_absolute_improvement": 0.001,
        "teacher_minimum_validation_cider_d": (
            QUALITY_THRESHOLD_OVERRIDE
        ),
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

clip_candidates = []
for config_path in KAGGLE_INPUT_ROOT.rglob("config.json"):
    try:
        config = read_json(config_path)
    except Exception:
        continue
    vision = config.get("vision_config", config)
    if (
        int(vision.get("hidden_size", -1)) == 768
        and int(vision.get("patch_size", -1)) == 32
        and int(vision.get("image_size", -1)) == 224
    ):
        weights = (
            list(config_path.parent.glob("*.safetensors"))
            + list(config_path.parent.glob("pytorch_model*.bin"))
        )
        if weights:
            clip_candidates.append(config_path.parent.resolve())
assert len(clip_candidates) == 1
CLIP_SOURCE_DIR = clip_candidates[0]

wheel_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "pycocoevalcap-1.2-py3-none-any.whl"
    )
)
assert len(wheel_matches) == 1, (
    "Hãy Add Input offline pycocoevalcap wheel."
)
PYCOCOEVALCAP_WHEEL = wheel_matches[0].resolve()

audit_path = AUDIT_DIR / "phase3d_cell01_environment_inputs.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3d_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "run_seed": SEED,
        "base_contract_development_seed": (
            BASE_CONTRACT_DEVELOPMENT_SEED
        ),
        "shared_initialization_seed": 42,
        "run_seed_contract": str(RUN_SEED_CONTRACT_PATH),
        "run_seed_contract_sha256": sha256_file(
            RUN_SEED_CONTRACT_PATH
        ),
        "phase3c_manifest_rows": len(phase3c_rows),
        "clip_source": str(CLIP_SOURCE_DIR),
        "metric_wheel": str(PYCOCOEVALCAP_WHEEL),
        "metric_wheel_sha256": sha256_file(PYCOCOEVALCAP_WHEEL),
        "gpu": torch.cuda.get_device_name(0),
        "strict_deterministic": True,
        "data_read": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3D CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher                   : A — CLIP ViT-B/32")
print("Seed                      : 2026")
print("Shared initialization seed: 42")
print("Matched S1 CIDEr-D         :", MATCHED_STUDENT_CIDER_D)
print("Teacher quality threshold :", QUALITY_THRESHOLD_OVERRIDE)
print("GPU                       :", torch.cuda.get_device_name(0))
print("Phase 03C                 : VERIFIED")
print("Teacher A training engine : READY")
print("Offline CLIP              : FOUND")
print("Metric wheel              : FOUND")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 8 cells passed")
print("Next                      : build Train/Validation datasets")

## Cell 2 — Train/Validation datasets và DataLoaders

### Công việc

- đọc Phase 1 Train/Validation artifacts;
- dựng 113.287 ảnh Train và 5.000 ảnh Validation;
- image-balanced sampling, một caption mỗi ảnh mỗi epoch;
- stateless train augmentation;
- tạo Validation caption và image datasets;
- không mở Test Parquet.

### Thời gian dự kiến

1–8 phút tùy Kaggle input.

### Đầu ra dự kiến

```text
Status                    : PASSED
Training images           : 113287
Validation captions       : 25010
Validation images         : 5000
Train repeat error        : 0.0
Progress                  : 2 / 8 cells passed
```


In [ ]:
CELL_STARTED_AT = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode


def exactly_one(root, pattern):
    matches = sorted(Path(root).rglob(pattern))
    assert len(matches) == 1, (
        f"Cần đúng một {pattern}, thấy {len(matches)}."
    )
    return matches[0].resolve()


phase1_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE1_FORMAL_DELIVERABLES_READY.json"
    )
)
assert len(phase1_matches) == 1
PHASE1_ROOT = phase1_matches[0].resolve().parent
TRAIN_PARQUET = exactly_one(PHASE1_ROOT, "mscoco_train.parquet")
VAL_PARQUET = exactly_one(PHASE1_ROOT, "mscoco_validation.parquet")
IMAGE_MANIFEST = exactly_one(
    PHASE1_ROOT, "mscoco_image_manifest.parquet"
)
TEST_PARQUET = exactly_one(PHASE1_ROOT, "mscoco_test.parquet")
VOCAB_PATH = exactly_one(
    PHASE1_ROOT, "word_vocabulary_minfreq5.json"
)

train_table = pq.read_table(
    TRAIN_PARQUET,
    columns=["image_id", "token_ids"],
)
validation_table = pq.read_table(
    VAL_PARQUET,
    columns=["image_id", "token_ids", "caption"],
)
manifest_table = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id",
        "experiment_split",
        "source_partition",
        "image_path",
    ],
)


def find_image_leaf(name, prefix):
    matches = []
    for directory in KAGGLE_INPUT_ROOT.rglob(name):
        if directory.is_dir() and next(
            directory.glob(prefix + "*.jpg"), None
        ) is not None:
            matches.append(directory.resolve())
    assert len(matches) == 1, (
        f"Cần đúng một {name} leaf, thấy {matches}."
    )
    return matches[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")
TRAIN_PATHS = {}
VAL_PATHS = {}
TEST_IDS = set()
for row in manifest_table.to_pylist():
    image_id = int(row["image_id"])
    split = row["experiment_split"]
    partition = row["source_partition"]
    file_name = Path(row["image_path"]).name
    if split == "train":
        leaf = TRAIN_LEAF if partition == "train2014" else VAL_LEAF
        path = leaf / file_name
        assert path.is_file()
        TRAIN_PATHS[image_id] = path
    elif split == "validation":
        path = VAL_LEAF / file_name
        assert path.is_file()
        VAL_PATHS[image_id] = path
    elif split == "held_out_test":
        TEST_IDS.add(image_id)

TRAIN_CAPTIONS = defaultdict(list)
for image_id, token_ids in zip(
    train_table["image_id"].to_pylist(),
    train_table["token_ids"].to_pylist(),
):
    TRAIN_CAPTIONS[int(image_id)].append(
        [int(token) for token in token_ids]
    )
VAL_RECORDS = [
    {
        "image_id": int(image_id),
        "token_ids": [int(token) for token in token_ids],
        "caption": str(caption),
    }
    for image_id, token_ids, caption in zip(
        validation_table["image_id"].to_pylist(),
        validation_table["token_ids"].to_pylist(),
        validation_table["caption"].to_pylist(),
    )
]
assert len(TRAIN_PATHS) == 113287
assert len(TRAIN_CAPTIONS) == 113287
assert len(VAL_PATHS) == 5000
assert len(VAL_RECORDS) == 25010
assert len(TEST_IDS) == 5000
assert not (set(TRAIN_PATHS) & TEST_IDS)
assert not (set(VAL_PATHS) & TEST_IDS)

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            224,
            scale=(0.8, 1.0),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ToTensor(),
        transforms.Normalize(CLIP_MEAN, CLIP_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.Resize(
            224,
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(CLIP_MEAN, CLIP_STD),
    ]
)


def deterministic_sample_seed(epoch, image_id, position):
    return (
        SEED * 1000003
        + int(epoch) * 9176
        + int(image_id) * 37
        + int(position) * 101
    ) % (2**31 - 1)


class TrainDataset(Dataset):
    def __init__(self):
        self.image_ids = sorted(TRAIN_PATHS)
        self.epoch = 0
        self.order = []
        self.set_epoch(0)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.order = list(self.image_ids)
        random.Random(SEED + self.epoch).shuffle(self.order)

    def __len__(self):
        return len(self.order)

    def __getitem__(self, position):
        image_id = self.order[position]
        captions = TRAIN_CAPTIONS[image_id]
        token_ids = captions[self.epoch % len(captions)]
        with Image.open(TRAIN_PATHS[image_id]) as image:
            image = image.convert("RGB")
            seed = deterministic_sample_seed(
                self.epoch, image_id, position
            )
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed(seed)
                pixel_values = train_transform(image)
        return image_id, pixel_values, token_ids


class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VAL_RECORDS)

    def __getitem__(self, index):
        record = VAL_RECORDS[index]
        with Image.open(VAL_PATHS[record["image_id"]]) as image:
            pixel_values = eval_transform(image.convert("RGB"))
        return (
            record["image_id"],
            pixel_values,
            record["token_ids"],
        )


class ValidationImageDataset(Dataset):
    def __init__(self):
        self.image_ids = sorted(VAL_PATHS)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        with Image.open(VAL_PATHS[image_id]) as image:
            pixel_values = eval_transform(image.convert("RGB"))
        return image_id, pixel_values


def caption_collate(rows):
    maximum = max(len(row[2]) for row in rows)
    full = torch.zeros(len(rows), maximum, dtype=torch.long)
    for index, row in enumerate(rows):
        sequence = torch.tensor(row[2], dtype=torch.long)
        full[index, : sequence.numel()] = sequence
    return {
        "image_ids": torch.tensor(
            [row[0] for row in rows], dtype=torch.long
        ),
        "pixel_values": torch.stack([row[1] for row in rows]),
        "decoder_input_ids": full[:, :-1],
        "decoder_attention_mask": full[:, :-1].ne(0),
        "labels": full[:, 1:],
    }


def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row[0] for row in rows], dtype=torch.long
        ),
        "pixel_values": torch.stack([row[1] for row in rows]),
    }


TRAIN_DATASET = TrainDataset()
VAL_CAPTION_DATASET = ValidationCaptionDataset()
VAL_IMAGE_DATASET = ValidationImageDataset()
sample_a = TRAIN_DATASET[0]
sample_b = TRAIN_DATASET[0]
repeat_error = (sample_a[1] - sample_b[1]).abs().max().item()
assert repeat_error == 0.0

audit_path = AUDIT_DIR / "phase3d_cell02_datasets.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3d_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "training_images": len(TRAIN_DATASET),
        "validation_captions": len(VAL_CAPTION_DATASET),
        "validation_images": len(VAL_IMAGE_DATASET),
        "train_repeat_error": repeat_error,
        "test_overlap": 0,
        "test_parquet_opened": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3D CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Training images           :", len(TRAIN_DATASET))
print("Training captions source  :", train_table.num_rows)
print("Validation captions       :", len(VAL_CAPTION_DATASET))
print("Validation images         :", len(VAL_IMAGE_DATASET))
print("Train repeat error        :", repeat_error)
print("Test overlap              : 0")
print("Test Parquet opened       : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 8 cells passed")
print("Next                      : reconstruct model and optimizer")

## Cell 3 — Model, staged optimizer và checkpoint state

### Công việc

- dựng CLIP vision encoder, adapter và shared decoder;
- strict-load initial adapter/decoder;
- dựng frozen/partial-unfreeze stages;
- tạo AdamW, warmup-cosine và GradScaler;
- định nghĩa atomic checkpoint state.

### Thời gian dự kiến

20–120 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Total parameters          : 93425248
Stage 1                   : frozen encoder
Total optimizer steps     : 35420
Initial strict load       : PASSED
Progress                  : 3 / 8 cells passed
```


In [ ]:
CELL_STARTED_AT = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel

DECODER_CONFIG = {
    "vocabulary_size": 10080,
    "pad_id": 0,
    "bos_id": 1,
    "eos_id": 2,
    "unk_id": 3,
    "d_model": 256,
    "nhead": 8,
    "num_layers": 3,
    "dim_feedforward": 1024,
    "dropout": 0.1,
    "activation": "gelu",
    "maximum_decoder_steps": 31,
}


class VisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(
            torch.zeros(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)
        nn.init.trunc_normal_(self.position_embedding, std=0.02)

    def forward(self, patches):
        return self.output_norm(
            self.projection(patches) + self.position_embedding
        )


class CaptionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.token_embedding = nn.Embedding(
            config["vocabulary_size"],
            config["d_model"],
            padding_idx=config["pad_id"],
        )
        self.position_embedding = nn.Embedding(
            config["maximum_decoder_steps"],
            config["d_model"],
        )
        self.embedding_dropout = nn.Dropout(config["dropout"])
        layer = nn.TransformerDecoderLayer(
            d_model=config["d_model"],
            nhead=config["nhead"],
            dim_feedforward=config["dim_feedforward"],
            dropout=config["dropout"],
            activation=config["activation"],
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=config["num_layers"]
        )
        self.final_norm = nn.LayerNorm(config["d_model"])
        self.output_bias = nn.Parameter(
            torch.zeros(config["vocabulary_size"])
        )

    def forward(self, token_ids, memory, attention_mask):
        batch, length = token_ids.shape
        positions = torch.arange(
            length, device=token_ids.device
        ).unsqueeze(0).expand(batch, -1)
        target = (
            self.token_embedding(token_ids) * math.sqrt(256)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=token_ids.device,
            ),
            diagonal=1,
        )
        hidden = self.transformer_decoder(
            target,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~attention_mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden,
            self.token_embedding.weight,
            self.output_bias,
        )


full_clip = CLIPModel.from_pretrained(
    CLIP_SOURCE_DIR, local_files_only=True
).to(device)
ENCODER = full_clip.vision_model
del full_clip
ADAPTER = VisualAdapter().to(device)
DECODER = CaptionDecoder(DECODER_CONFIG).to(device)

try:
    from torch.torch_version import TorchVersion
    safe_context = torch.serialization.safe_globals([TorchVersion])
except Exception:
    from contextlib import nullcontext
    safe_context = nullcontext()
with safe_context:
    INITIAL = torch.load(
        INITIAL_LIGHT_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
ADAPTER.load_state_dict(INITIAL["adapter_state_dict"], strict=True)
DECODER.load_state_dict(INITIAL["decoder_state_dict"], strict=True)

PHYSICAL_BATCH = 16
ACCUMULATION = 4
EFFECTIVE_BATCH = 64
STEPS_PER_EPOCH = math.ceil(113287 / EFFECTIVE_BATCH)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)
BACKWARD_DIVISOR = 1024.0


def encoder_trainable_parameters():
    parameters = []
    for layer in ENCODER.encoder.layers[-2:]:
        parameters.extend(list(layer.parameters()))
    parameters.extend(list(ENCODER.post_layernorm.parameters()))
    unique, seen = [], set()
    for parameter in parameters:
        if id(parameter) not in seen:
            unique.append(parameter)
            seen.add(id(parameter))
    return unique


ENCODER_TRAINABLE = encoder_trainable_parameters()
ENCODER_TRAINABLE_IDS = {
    id(parameter) for parameter in ENCODER_TRAINABLE
}
ENCODER_INITIAL_CANDIDATE_STATE = {
    name: value.detach().cpu().clone()
    for name, value in ENCODER.state_dict().items()
    if name
    in {
        parameter_name
        for parameter_name, parameter in ENCODER.named_parameters()
        if id(parameter) in ENCODER_TRAINABLE_IDS
    }
}
assert ENCODER_INITIAL_CANDIDATE_STATE


def set_stage(epoch):
    for parameter in ENCODER.parameters():
        parameter.requires_grad = False
    if int(epoch) >= 6:
        for parameter in ENCODER_TRAINABLE:
            parameter.requires_grad = True
        return "partial_unfreeze"
    return "frozen_encoder"


def lr_factor(step):
    if step < WARMUP_STEPS:
        return max((step + 1) / WARMUP_STEPS, 1e-8)
    progress = (
        (step - WARMUP_STEPS)
        / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    )
    return 0.05 + 0.95 * 0.5 * (
        1 + math.cos(math.pi * progress)
    )


def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": ENCODER_TRAINABLE, "lr": 1e-5},
            {"params": ADAPTER.parameters(), "lr": 1e-4},
            {"params": DECODER.parameters(), "lr": 3e-4},
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lr_factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", init_scale=1024.0
    )
    return optimizer, scheduler, scaler


def restore_initial():
    encoder_state = ENCODER.state_dict()
    encoder_state.update(ENCODER_INITIAL_CANDIDATE_STATE)
    ENCODER.load_state_dict(encoder_state, strict=True)
    ADAPTER.load_state_dict(INITIAL["adapter_state_dict"], strict=True)
    DECODER.load_state_dict(INITIAL["decoder_state_dict"], strict=True)
    set_stage(1)


set_stage(1)
OPTIMIZER, SCHEDULER, SCALER = build_optimizer_state()
total_parameters = (
    sum(p.numel() for p in ENCODER.parameters())
    + sum(p.numel() for p in ADAPTER.parameters())
    + sum(p.numel() for p in DECODER.parameters())
)
assert total_parameters == 93425248
assert len(OPTIMIZER.state) == 0

audit_path = AUDIT_DIR / "phase3d_cell03_model_optimizer.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3d_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "total_parameters": total_parameters,
        "initial_checkpoint_sha256": sha256_file(
            INITIAL_LIGHT_CHECKPOINT
        ),
        "encoder_initial_candidate_tensors": len(
            ENCODER_INITIAL_CANDIDATE_STATE
        ),
        "stage_1": "frozen_encoder",
        "stage_2": "partial_unfreeze_last2",
        "steps_per_epoch": STEPS_PER_EPOCH,
        "total_steps": TOTAL_STEPS,
        "warmup_steps": WARMUP_STEPS,
        "optimizer_state_empty": True,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3D CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Total parameters          :", total_parameters)
print("Stage 1                   : frozen encoder")
print("Stage 2                   : unfreeze last two blocks")
print("Physical/effective batch  : 16 / 64")
print("Optimizer steps/epoch     :", STEPS_PER_EPOCH)
print("Total optimizer steps     :", TOTAL_STEPS)
print("Warmup steps              :", WARMUP_STEPS)
print("Initial strict load       : PASSED")
print("Optimizer state           : EMPTY")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 8 cells passed")
print("Next                      : initialize COCO metrics")

## Cell 4 — Offline COCO metric backend và Validation references

### Công việc

- cài `pycocoevalcap==1.2` từ wheel offline nếu cần;
- dựng references cho 5.000 Validation images;
- smoke test BLEU, METEOR, ROUGE-L và CIDEr-D;
- không đọc Test captions.

### Thời gian dự kiến

10–90 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Validation references     : 25010
Metric backend            : pycocoevalcap 1.2
Metrics                   : 5 / 5 PASSED
Progress                  : 4 / 8 cells passed
```


In [ ]:
CELL_STARTED_AT = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except ImportError:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-deps",
            str(PYCOCOEVALCAP_WHEEL),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge

REFERENCES = defaultdict(list)
for record in VAL_RECORDS:
    REFERENCES[record["image_id"]].append(
        {"caption": record["caption"]}
    )
assert len(REFERENCES) == 5000
assert sum(len(values) for values in REFERENCES.values()) == 25010


def compute_metrics(reference_text, hypothesis_text):
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    tokenizer = PTBTokenizer()
    references_tokenized = tokenizer.tokenize(reference_text)
    hypotheses_tokenized = tokenizer.tokenize(hypothesis_text)
    bleu_scores, _ = Bleu(4).compute_score(
        references_tokenized, hypotheses_tokenized
    )
    meteor_backend = Meteor()
    try:
        meteor_score, _ = meteor_backend.compute_score(
            references_tokenized, hypotheses_tokenized
        )
    finally:
        if hasattr(meteor_backend, "close"):
            meteor_backend.close()
    rouge_score, _ = Rouge().compute_score(
        references_tokenized, hypotheses_tokenized
    )
    cider_score, _ = Cider().compute_score(
        references_tokenized, hypotheses_tokenized
    )
    return {
        "BLEU-1": float(bleu_scores[0]),
        "BLEU-4": float(bleu_scores[3]),
        "METEOR": float(meteor_score),
        "ROUGE-L": float(rouge_score),
        "CIDEr-D": float(cider_score),
    }


smoke_ids = sorted(REFERENCES)[:32]
smoke_references = {
    image_id: REFERENCES[image_id] for image_id in smoke_ids
}
smoke_hypotheses = {
    image_id: [REFERENCES[image_id][0]]
    for image_id in smoke_ids
}
smoke_metrics = compute_metrics(
    smoke_references, smoke_hypotheses
)
assert len(smoke_metrics) == 5
assert all(math.isfinite(value) for value in smoke_metrics.values())

vocab = read_json(VOCAB_PATH)
ID_TO_TOKEN = list(vocab["id_to_token"])
assert len(ID_TO_TOKEN) == 10080

audit_path = AUDIT_DIR / "phase3d_cell04_metric_backend.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3d_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "metric_backend": "pycocoevalcap_1.2",
        "offline_wheel_sha256": sha256_file(
            PYCOCOEVALCAP_WHEEL
        ),
        "validation_images": len(REFERENCES),
        "validation_references": sum(
            len(values) for values in REFERENCES.values()
        ),
        "smoke_metrics": smoke_metrics,
        "test_caption_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3D CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Metric backend            : pycocoevalcap 1.2")
print("Offline installation      : READY")
print("Validation images         :", len(REFERENCES))
print("Validation references     :", sum(
    len(values) for values in REFERENCES.values()
))
print("Metrics                   : 5 / 5 PASSED")
print("All metrics finite        : True")
print("Test caption read         : False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 8 cells passed")
print("Next                      : validate Beam-3 generation engine")

## Cell 5 — Teacher-forced Validation và Beam-3 engine

### Công việc

- định nghĩa teacher-forced CE;
- định nghĩa batched Beam-3;
- cấm PAD, BOS và UNK khi sinh;
- xác minh generation lặp lại chính xác;
- smoke test trên Validation subset.

### Thời gian dự kiến

20–120 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Teacher-forced CE         : finite
Beam size                 : 3
Token repeat exact        : True
Metrics wired             : 5 / 5
Progress                  : 5 / 8 cells passed
```


In [ ]:
CELL_STARTED_AT = time.perf_counter()


def move_batch(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if torch.is_tensor(value)
            else value
        )
        for key, value in batch.items()
    }


def encode_images(pixel_values, requires_gradient=False):
    context = (
        torch.enable_grad()
        if requires_gradient
        else torch.no_grad()
    )
    with context:
        output = ENCODER(
            pixel_values=pixel_values
        ).last_hidden_state[:, 1:, :]
    return ADAPTER(output)


def decode_ids(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words).strip()


def beam_generate(memory, beam_size=3):
    batch_size = memory.shape[0]
    vocabulary_size = 10080
    sequences = torch.full(
        (batch_size, beam_size, 1),
        1,
        dtype=torch.long,
        device=memory.device,
    )
    scores = torch.full(
        (batch_size, beam_size),
        float("-inf"),
        device=memory.device,
    )
    scores[:, 0] = 0.0
    finished = torch.zeros(
        batch_size,
        beam_size,
        dtype=torch.bool,
        device=memory.device,
    )
    expanded_memory = (
        memory.unsqueeze(1)
        .expand(-1, beam_size, -1, -1)
        .reshape(batch_size * beam_size, 49, 256)
    )
    for step in range(31):
        flat_sequences = sequences.reshape(
            batch_size * beam_size, -1
        )
        mask = flat_sequences.ne(0)
        logits = DECODER(
            flat_sequences,
            expanded_memory,
            mask,
        )[:, -1, :]
        logits[:, [0, 1, 3]] = float("-inf")
        if step < 2:
            logits[:, 2] = float("-inf")
        log_probabilities = F.log_softmax(logits, dim=-1)
        flat_finished = finished.reshape(-1)
        if flat_finished.any():
            log_probabilities[flat_finished] = float("-inf")
            log_probabilities[flat_finished, 2] = 0.0
        candidates = (
            scores.reshape(-1, 1) + log_probabilities
        ).reshape(batch_size, beam_size * vocabulary_size)
        top_scores, top_indices = candidates.topk(
            beam_size, dim=-1
        )
        previous_beam = top_indices // vocabulary_size
        next_token = top_indices % vocabulary_size
        gather_index = previous_beam.unsqueeze(-1).expand(
            -1, -1, sequences.shape[-1]
        )
        sequences = torch.gather(
            sequences, 1, gather_index
        )
        sequences = torch.cat(
            [sequences, next_token.unsqueeze(-1)], dim=-1
        )
        finished = torch.gather(
            finished, 1, previous_beam
        ) | next_token.eq(2)
        scores = top_scores
        if finished.all():
            break
    lexical_lengths = (
        sequences.ne(0).sum(-1).float() - 1.0
    ).clamp_min(1.0)
    normalized = scores / lexical_lengths.pow(0.7)
    best_beam = normalized.argmax(-1)
    selected = sequences[
        torch.arange(batch_size, device=memory.device),
        best_beam,
    ]
    return selected


ENCODER.eval()
ADAPTER.eval()
DECODER.eval()
smoke_caption_loader = DataLoader(
    VAL_CAPTION_DATASET,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    collate_fn=caption_collate,
)
smoke_caption_batch = move_batch(next(iter(smoke_caption_loader)))
with torch.inference_mode():
    smoke_memory = encode_images(
        smoke_caption_batch["pixel_values"]
    )
    smoke_logits = DECODER(
        smoke_caption_batch["decoder_input_ids"],
        smoke_memory,
        smoke_caption_batch["decoder_attention_mask"],
    )
    smoke_ce = F.cross_entropy(
        smoke_logits.reshape(-1, 10080),
        smoke_caption_batch["labels"].reshape(-1),
        ignore_index=0,
    )
assert torch.isfinite(smoke_ce)

smoke_image_loader = DataLoader(
    VAL_IMAGE_DATASET,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    collate_fn=image_collate,
)
smoke_image_batch = move_batch(next(iter(smoke_image_loader)))
with torch.inference_mode():
    image_memory = encode_images(
        smoke_image_batch["pixel_values"]
    )
    generated_1 = beam_generate(image_memory)
    generated_2 = beam_generate(image_memory)
assert torch.equal(generated_1, generated_2)
smoke_captions = [
    decode_ids(row.tolist()) for row in generated_1
]

audit_path = AUDIT_DIR / "phase3d_cell05_validation_beam.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3d_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_forced_smoke_ce": float(smoke_ce.item()),
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_decoder_steps": 31,
        "token_repeat_exact": True,
        "smoke_captions": smoke_captions,
        "metrics_wired": 5,
        "optimizer_step_performed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3D CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher-forced CE         :", f"{smoke_ce.item():.8f}")
print("Beam size / penalty       : 3 / 0.7")
print("Maximum decoder steps     : 31")
print("Token repeat exact        : True")
print("Metrics wired             : 5 / 5")
print("Smoke captions            :", smoke_captions)
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 8 cells passed")
print("Next                      : build production trainer and pilot")

## Cell 6 — Production trainer, checkpoint/resume và discarded pilot

### Công việc

- định nghĩa token-balanced optimizer step;
- định nghĩa full Validation;
- định nghĩa atomic last/best checkpoint;
- hỗ trợ resume ở optimizer-step boundary;
- chạy discarded hai-step production pilot;
- phục hồi initial state.

### Thời gian dự kiến

30–180 giây.

### Đầu ra dự kiến

```text
Status                    : PASSED
Production trainer        : READY
Discarded pilot steps     : 2 / 2
Atomic checkpoint         : PASSED
Initial state restored    : True
Progress                  : 6 / 8 cells passed
```


In [ ]:
CELL_STARTED_AT = time.perf_counter()
LAST_CHECKPOINT = CHECKPOINT_DIR / "teacher_a_seed2026_last.pt"
BEST_CHECKPOINT = CHECKPOINT_DIR / "teacher_a_seed2026_best.pt"
STEP_LOG = LOG_DIR / "teacher_a_seed2026_optimizer_steps.jsonl"
EPOCH_HISTORY = LOG_DIR / "teacher_a_seed2026_epoch_history.csv"


def training_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch - 1)
    subset = torch.utils.data.Subset(
        TRAIN_DATASET,
        list(range(cursor, len(TRAIN_DATASET))),
    )
    return DataLoader(
        subset,
        batch_size=PHYSICAL_BATCH,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        collate_fn=caption_collate,
    )


def optimizer_step(batches, optimizer, scheduler, scaler):
    optimizer.zero_grad(set_to_none=True)
    stage_has_encoder_gradient = any(
        parameter.requires_grad
        for parameter in ENCODER.parameters()
    )
    # Keep every frozen CLIP block deterministic.  In stage 2, only the
    # explicitly trainable tail blocks and post-LayerNorm enter train mode.
    ENCODER.eval()
    if stage_has_encoder_gradient:
        for layer in ENCODER.encoder.layers[-2:]:
            layer.train()
        ENCODER.post_layernorm.train()
    ADAPTER.train()
    DECODER.train()
    loss_sum_total = 0.0
    valid_total = 0
    scale_before = scaler.get_scale()
    for raw_batch in batches:
        batch = move_batch(raw_batch)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            if stage_has_encoder_gradient:
                patches = ENCODER(
                    pixel_values=batch["pixel_values"]
                ).last_hidden_state[:, 1:, :]
            else:
                with torch.no_grad():
                    patches = ENCODER(
                        pixel_values=batch["pixel_values"]
                    ).last_hidden_state[:, 1:, :]
            memory = ADAPTER(patches)
            logits = DECODER(
                batch["decoder_input_ids"],
                memory,
                batch["decoder_attention_mask"],
            )
            loss_sum = F.cross_entropy(
                logits.reshape(-1, 10080),
                batch["labels"].reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
        valid = int(batch["labels"].ne(0).sum().item())
        scaler.scale(loss_sum / 1024.0).backward()
        loss_sum_total += float(loss_sum.detach().item())
        valid_total += valid
    scaler.unscale_(optimizer)
    correction = 1024.0 / valid_total
    parameters_with_grad = []
    for group in optimizer.param_groups:
        for parameter in group["params"]:
            if parameter.grad is not None:
                parameter.grad.mul_(correction)
                parameters_with_grad.append(parameter)
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        parameters_with_grad, 1.0
    )
    gradient_finite = bool(
        torch.isfinite(gradient_norm).item()
    )
    # GradScaler đã ghi nhận found_inf khi unscale_. Không assert tại đây:
    # scaler.step() phải được gọi để tự bỏ optimizer step không hữu hạn,
    # scaler.update() giảm scale, rồi vòng ngoài retry đúng microbatch.
    scaler.step(optimizer)
    scaler.update()
    overflow = scaler.get_scale() < scale_before
    if not gradient_finite:
        assert overflow, (
            "Gradient norm không hữu hạn nhưng GradScaler không phát hiện "
            "overflow; dừng để tránh cập nhật trọng số không hợp lệ."
        )
    if not overflow:
        scheduler.step()
    return {
        "mean_token_ce": loss_sum_total / valid_total,
        "valid_tokens": valid_total,
        "gradient_norm": float(gradient_norm.item()),
        "gradient_finite": gradient_finite,
        "overflow": overflow,
        "scaler_scale_before": float(scale_before),
        "scaler_scale_after": float(scaler.get_scale()),
        "images": sum(
            int(batch["pixel_values"].shape[0])
            for batch in batches
        ),
    }


def full_validation(epoch):
    ENCODER.eval()
    ADAPTER.eval()
    DECODER.eval()
    caption_loader = DataLoader(
        VAL_CAPTION_DATASET,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=caption_collate,
    )
    loss_sum, valid_tokens = 0.0, 0
    with torch.inference_mode():
        for raw_batch in caption_loader:
            batch = move_batch(raw_batch)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                memory = encode_images(batch["pixel_values"])
                logits = DECODER(
                    batch["decoder_input_ids"],
                    memory,
                    batch["decoder_attention_mask"],
                )
                batch_loss = F.cross_entropy(
                    logits.reshape(-1, 10080),
                    batch["labels"].reshape(-1),
                    ignore_index=0,
                    reduction="sum",
                )
            loss_sum += float(batch_loss.item())
            valid_tokens += int(batch["labels"].ne(0).sum().item())
    mean_ce = loss_sum / valid_tokens

    image_loader = DataLoader(
        VAL_IMAGE_DATASET,
        batch_size=16,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=image_collate,
    )
    predictions = {}
    with torch.inference_mode():
        for raw_batch in image_loader:
            batch = move_batch(raw_batch)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                memory = encode_images(batch["pixel_values"])
                generated = beam_generate(memory, beam_size=3)
            for image_id, token_row in zip(
                batch["image_ids"].tolist(),
                generated.tolist(),
            ):
                predictions[int(image_id)] = [
                    {"caption": decode_ids(token_row)}
                ]
    assert len(predictions) == 5000
    metrics = compute_metrics(dict(REFERENCES), predictions)
    result = {
        "epoch": int(epoch),
        "teacher_forced_ce": mean_ce,
        "metrics": metrics,
        "prediction_count": len(predictions),
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    result_path = (
        VALIDATION_DIR / f"epoch_{epoch:02d}_validation.json"
    )
    write_json(
        result_path,
        {
            **result,
            "predictions": {
                str(key): value for key, value in predictions.items()
            },
        },
    )
    return result


def candidate_state():
    candidate_ids = {id(parameter) for parameter in ENCODER_TRAINABLE}
    candidate_names = {
        name
        for name, parameter in ENCODER.named_parameters()
        if id(parameter) in candidate_ids
    }
    return {
        name: value.detach().cpu()
        for name, value in ENCODER.state_dict().items()
        if name in candidate_names
    }


def atomic_save(path, payload):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_checkpoint(
    path,
    optimizer,
    scheduler,
    scaler,
    progress,
    history,
    best_result,
):
    atomic_save(
        path,
        {
            "schema_version": "phase3_teacher_a_seed2026_checkpoint_v1",
            "teacher_id": "A",
            "seed": 2026,
            "encoder_candidate_state": candidate_state(),
            "adapter_state": {
                key: value.detach().cpu()
                for key, value in ADAPTER.state_dict().items()
            },
            "decoder_state": {
                key: value.detach().cpu()
                for key, value in DECODER.state_dict().items()
            },
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "progress": progress,
            "history": history,
            "best_result": best_result,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
            "training_contract_sha256": sha256_file(
                TRAINING_CONTRACT_PATH
            ),
            "run_seed_contract_sha256": sha256_file(
                RUN_SEED_CONTRACT_PATH
            ),
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )


def load_checkpoint(path, optimizer, scheduler, scaler):
    try:
        from torch.torch_version import TorchVersion
        checkpoint_context = torch.serialization.safe_globals(
            [TorchVersion]
        )
    except ImportError:
        checkpoint_context = nullcontext()
    with checkpoint_context:
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
    assert checkpoint["test_set_status"] == "sealed"
    assert checkpoint["test_set_used"] is False
    encoder_state = ENCODER.state_dict()
    encoder_state.update(checkpoint["encoder_candidate_state"])
    ENCODER.load_state_dict(encoder_state, strict=True)
    ADAPTER.load_state_dict(checkpoint["adapter_state"], strict=True)
    DECODER.load_state_dict(checkpoint["decoder_state"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])
    torch.set_rng_state(checkpoint["cpu_rng"])
    torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
    return (
        checkpoint["progress"],
        checkpoint["history"],
        checkpoint["best_result"],
    )


restore_initial()
pilot_optimizer, pilot_scheduler, pilot_scaler = (
    build_optimizer_state()
)
pilot_loader = training_loader(1, 0)
pilot_iterator = iter(pilot_loader)
pilot_results = []
for _ in range(2):
    pilot_results.append(
        optimizer_step(
            [next(pilot_iterator) for _ in range(4)],
            pilot_optimizer,
            pilot_scheduler,
            pilot_scaler,
        )
    )
assert all(
    math.isfinite(item["mean_token_ce"])
    and math.isfinite(item["gradient_norm"])
    and not item["overflow"]
    for item in pilot_results
)
temporary_checkpoint = CHECKPOINT_DIR / "temporary_production_pilot.pt"
pilot_progress = {
    "epoch": 1,
    "cursor": 128,
    "global_step": 2,
    "overflow_retry_count": 0,
    "epoch_loss_sum": sum(
        item["mean_token_ce"] * item["valid_tokens"]
        for item in pilot_results
    ),
    "epoch_tokens": sum(
        item["valid_tokens"] for item in pilot_results
    ),
    "epoch_images": sum(
        item["images"] for item in pilot_results
    ),
}
save_checkpoint(
    temporary_checkpoint,
    pilot_optimizer,
    pilot_scheduler,
    pilot_scaler,
    pilot_progress,
    [],
    None,
)
assert temporary_checkpoint.is_file()


def clone_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {
            key: clone_to_cpu(item)
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [clone_to_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_to_cpu(item) for item in value)
    return copy.deepcopy(value)


def nested_exact(left, right):
    if torch.is_tensor(left) and torch.is_tensor(right):
        return torch.equal(
            left.detach().cpu(), right.detach().cpu()
        )
    if isinstance(left, dict) and isinstance(right, dict):
        return (
            left.keys() == right.keys()
            and all(
                nested_exact(left[key], right[key])
                for key in left
            )
        )
    if isinstance(left, (list, tuple)) and isinstance(
        right, (list, tuple)
    ):
        return (
            len(left) == len(right)
            and all(
                nested_exact(a, b)
                for a, b in zip(left, right)
            )
        )
    return left == right


expected_encoder = clone_to_cpu(candidate_state())
expected_adapter = clone_to_cpu(ADAPTER.state_dict())
expected_decoder = clone_to_cpu(DECODER.state_dict())
expected_optimizer = clone_to_cpu(pilot_optimizer.state_dict())
expected_scheduler = clone_to_cpu(pilot_scheduler.state_dict())
expected_scaler = clone_to_cpu(pilot_scaler.state_dict())
expected_cpu_rng = torch.get_rng_state().clone()
expected_cuda_rng = [
    value.clone() for value in torch.cuda.get_rng_state_all()
]

restore_initial()
resume_optimizer, resume_scheduler, resume_scaler = (
    build_optimizer_state()
)
loaded_progress, loaded_history, loaded_best = load_checkpoint(
    temporary_checkpoint,
    resume_optimizer,
    resume_scheduler,
    resume_scaler,
)
assert loaded_progress == pilot_progress
assert loaded_history == []
assert loaded_best is None
assert nested_exact(candidate_state(), expected_encoder)
assert nested_exact(ADAPTER.state_dict(), expected_adapter)
assert nested_exact(DECODER.state_dict(), expected_decoder)
assert nested_exact(
    resume_optimizer.state_dict(), expected_optimizer
)
assert nested_exact(
    resume_scheduler.state_dict(), expected_scheduler
)
assert nested_exact(resume_scaler.state_dict(), expected_scaler)
assert torch.equal(torch.get_rng_state(), expected_cpu_rng)
assert nested_exact(
    torch.cuda.get_rng_state_all(), expected_cuda_rng
)
resume_exact = True

temporary_checkpoint.unlink()
restore_initial()
assert not temporary_checkpoint.exists()
assert nested_exact(
    candidate_state(), ENCODER_INITIAL_CANDIDATE_STATE
)
assert nested_exact(
    ADAPTER.state_dict(), INITIAL["adapter_state_dict"]
)
assert nested_exact(
    DECODER.state_dict(), INITIAL["decoder_state_dict"]
)

audit_path = AUDIT_DIR / "phase3d_cell06_production_trainer.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3d_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "discarded_pilot_steps": 2,
        "discarded_pilot_results": pilot_results,
        "atomic_checkpoint": True,
        "resume_load_exact": resume_exact,
        "temporary_checkpoint_deleted": True,
        "initial_state_restored": True,
        "full_validation_run": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3D CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Production trainer        : READY")
print("Discarded pilot steps     : 2 / 2 PASSED")
print("Atomic checkpoint         : PASSED")
print("Mid-epoch resume          : PASSED")
print("Full Validation run       : False")
print("Temporary checkpoint      : DELETED")
print("Initial state restored    : True")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 8 cells passed")
print("Next                      : launch or resume full training")

## Cell 7 — Full Teacher A seed-2026 training

### Công việc

- tự động resume từ `last` checkpoint nếu có;
- train tối đa 20 epoch;
- Stage 1 frozen encoder, Stage 2 partial unfreeze;
- checkpoint mỗi 250 optimizer steps;
- full Validation mỗi epoch;
- chọn best bằng CIDEr-D, sau đó BLEU-4, CE và epoch sớm;
- chạy đủ 20 epoch theo locked contract;
- không mở Test Set.

### Thời gian dự kiến

Khoảng 8–14 giờ trên T4; có thể cần nhiều Kaggle session vì full
Beam-3 Validation chạy sau mỗi epoch.

### Đầu ra dự kiến

```text
Status                    : PASSED
Best Validation CIDEr-D   : finite
Best checkpoint           : VERIFIED
Test Set status           : SEALED
Progress                  : 7 / 8 cells passed
```


In [ ]:
CELL_STARTED_AT = time.perf_counter()
COMPLETE_MARKER = OUTPUT_ROOT / "TEACHER_A_SEED2026_FULL_RUN_COMPLETE.json"
assert not COMPLETE_MARKER.exists(), (
    "Run đã có COMPLETE marker. Không huấn luyện lặp."
)

restore_initial()
OPTIMIZER, SCHEDULER, SCALER = build_optimizer_state()
external_last = sorted(
    KAGGLE_INPUT_ROOT.rglob("teacher_a_seed2026_last.pt")
)
if LAST_CHECKPOINT.exists():
    resume_path = LAST_CHECKPOINT
    run_mode = "local_resume"
elif len(external_last) == 1:
    resume_path = external_last[0]
    run_mode = "external_resume"
elif len(external_last) > 1:
    raise AssertionError(
        f"Nhiều resume checkpoints: {external_last}"
    )
else:
    resume_path = None
    run_mode = "new_run"

if resume_path is not None:
    progress, history, best_result = load_checkpoint(
        resume_path, OPTIMIZER, SCHEDULER, SCALER
    )
else:
    progress = {
        "epoch": 1,
        "cursor": 0,
        "global_step": 0,
        "overflow_retry_count": 0,
        "epoch_loss_sum": 0.0,
        "epoch_tokens": 0,
        "epoch_images": 0,
    }
    history = []
    best_result = None

# Backward-compatible default for checkpoints created before the retry counter
# was added. Phase 03D enforces at most 10 retries for one optimizer step.
progress.setdefault("overflow_retry_count", 0)
progress.setdefault("epoch_loss_sum", 0.0)
progress.setdefault("epoch_tokens", 0)
progress.setdefault("epoch_images", 0)


def is_better(candidate, current):
    if current is None:
        return True
    candidate_metrics = candidate["metrics"]
    current_metrics = current["metrics"]
    key_candidate = (
        candidate_metrics["CIDEr-D"],
        candidate_metrics["BLEU-4"],
        -candidate["teacher_forced_ce"],
        -candidate["epoch"],
    )
    key_current = (
        current_metrics["CIDEr-D"],
        current_metrics["BLEU-4"],
        -current["teacher_forced_ce"],
        -current["epoch"],
    )
    return key_candidate > key_current


training_started = time.perf_counter()
while progress["epoch"] <= MAX_EPOCHS:
    epoch = int(progress["epoch"])
    stage = set_stage(epoch)
    loader = training_loader(epoch, int(progress["cursor"]))
    iterator = iter(loader)
    epoch_loss_sum = float(progress["epoch_loss_sum"])
    epoch_tokens = int(progress["epoch_tokens"])
    epoch_images = int(progress["epoch_images"])
    while True:
        microbatches = []
        for _ in range(ACCUMULATION):
            try:
                microbatches.append(next(iterator))
            except StopIteration:
                break
        if not microbatches:
            break
        retries_for_step = 0
        while True:
            result = optimizer_step(
                microbatches, OPTIMIZER, SCHEDULER, SCALER
            )
            if not result["overflow"]:
                break
            retries_for_step += 1
            progress["overflow_retry_count"] += 1
            print(
                "FP16 overflow retry       : "
                f"epoch={epoch:02d} | "
                f"cursor={progress['cursor']:06d} | "
                f"retry={retries_for_step}/10 | "
                f"scale={result['scaler_scale_before']}"
                f"->{result['scaler_scale_after']}"
            )
            assert retries_for_step <= 10, (
                "Vượt FP16 overflow retry limit=10 tại "
                f"epoch={epoch}, cursor={progress['cursor']}."
            )
        progress["global_step"] += 1
        progress["cursor"] += result["images"]
        epoch_loss_sum += (
            result["mean_token_ce"] * result["valid_tokens"]
        )
        epoch_tokens += result["valid_tokens"]
        epoch_images += result["images"]
        progress["epoch_loss_sum"] = epoch_loss_sum
        progress["epoch_tokens"] = epoch_tokens
        progress["epoch_images"] = epoch_images
        with open(STEP_LOG, "a", encoding="utf-8") as target:
            target.write(
                json.dumps(
                    {
                        "epoch": epoch,
                        "stage": stage,
                        "cursor": progress["cursor"],
                        "global_step": progress["global_step"],
                        "overflow_retries_for_step": retries_for_step,
                        "overflow_retry_count": progress[
                            "overflow_retry_count"
                        ],
                        **result,
                        "test_set_used": False,
                    },
                    sort_keys=True,
                )
                + "\n"
            )
        if (
            progress["global_step"] > 0
            and progress["global_step"] % 250 == 0
        ):
            save_checkpoint(
                LAST_CHECKPOINT,
                OPTIMIZER,
                SCHEDULER,
                SCALER,
                dict(progress),
                history,
                best_result,
            )
        if progress["cursor"] % 6400 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"cursor={progress['cursor']:06d} | "
                f"step={progress['global_step']:05d} | "
                f"loss={result['mean_token_ce']:.6f} | "
                f"stage={stage}"
            )

    assert epoch_images > 0
    validation = full_validation(epoch)
    epoch_row = {
        "epoch": epoch,
        "stage": stage,
        "training_mean_token_ce": (
            epoch_loss_sum / epoch_tokens
        ),
        "training_images": epoch_images,
        **validation,
    }
    history.append(epoch_row)
    improved = is_better(validation, best_result)
    if improved:
        best_result = validation

    with open(
        EPOCH_HISTORY,
        "w",
        encoding="utf-8",
        newline="",
    ) as target:
        fieldnames = [
            "epoch",
            "stage",
            "training_mean_token_ce",
            "training_images",
            "teacher_forced_ce",
            "BLEU-1",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
            "CIDEr-D",
        ]
        writer = csv.DictWriter(target, fieldnames=fieldnames)
        writer.writeheader()
        for row in history:
            writer.writerow(
                {
                    "epoch": row["epoch"],
                    "stage": row["stage"],
                    "training_mean_token_ce": row[
                        "training_mean_token_ce"
                    ],
                    "training_images": row["training_images"],
                    "teacher_forced_ce": row["teacher_forced_ce"],
                    **row["metrics"],
                }
            )

    progress = {
        "epoch": epoch + 1,
        "cursor": 0,
        "global_step": progress["global_step"],
        "overflow_retry_count": progress[
            "overflow_retry_count"
        ],
        "epoch_loss_sum": 0.0,
        "epoch_tokens": 0,
        "epoch_images": 0,
    }
    save_checkpoint(
        LAST_CHECKPOINT,
        OPTIMIZER,
        SCHEDULER,
        SCALER,
        dict(progress),
        history,
        best_result,
    )
    if improved:
        save_checkpoint(
            BEST_CHECKPOINT,
            OPTIMIZER,
            SCHEDULER,
            SCALER,
            dict(progress),
            history,
            best_result,
        )
    print(
        f"epoch={epoch:02d} | "
        f"val_CE={validation['teacher_forced_ce']:.6f} | "
        f"CIDEr-D={validation['metrics']['CIDEr-D']:.6f} | "
        f"BLEU-4={validation['metrics']['BLEU-4']:.6f} | "
        f"best={best_result['metrics']['CIDEr-D']:.6f}"
    )
training_seconds = time.perf_counter() - training_started
assert BEST_CHECKPOINT.is_file()
assert LAST_CHECKPOINT.is_file()
assert history
assert best_result is not None
assert len(history) == MAX_EPOCHS
assert all(
    math.isfinite(float(value))
    for value in best_result["metrics"].values()
)
assert math.isfinite(float(best_result["teacher_forced_ce"]))

quality_threshold = float(
    QUALITY_THRESHOLD_OVERRIDE
)
assert math.isclose(
    quality_threshold,
    MATCHED_STUDENT_CIDER_D + 0.001,
    rel_tol=0.0,
    abs_tol=1e-12,
)
quality_gate_passed = (
    best_result["metrics"]["CIDEr-D"] >= quality_threshold
)

audit_path = AUDIT_DIR / "phase3d_cell07_full_training.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3d_cell07_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "run_mode": run_mode,
        "completed_epochs": len(history),
        "required_epochs": MAX_EPOCHS,
        "global_optimizer_steps": progress["global_step"],
        "overflow_retry_count": progress[
            "overflow_retry_count"
        ],
        "best_result": best_result,
        "matched_student_seed": SEED,
        "matched_student_cider_d": MATCHED_STUDENT_CIDER_D,
        "quality_threshold": quality_threshold,
        "quality_gate_passed": quality_gate_passed,
        "training_seconds": training_seconds,
        "last_checkpoint_sha256": sha256_file(LAST_CHECKPOINT),
        "best_checkpoint_sha256": sha256_file(BEST_CHECKPOINT),
        "selection_split": "validation",
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3D CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Run mode                  :", run_mode)
print("Completed epochs          :", len(history))
print("Global optimizer steps    :", progress["global_step"])
print("Best epoch                :", best_result["epoch"])
print("Best Validation CE        :", best_result["teacher_forced_ce"])
print("Best Validation CIDEr-D   :", best_result["metrics"]["CIDEr-D"])
print("Best Validation BLEU-4    :", best_result["metrics"]["BLEU-4"])
print("Teacher quality threshold :", quality_threshold)
print("Teacher quality gate      :", "PASSED" if quality_gate_passed else "NOT PASSED")
print("Last checkpoint           : VERIFIED")
print("Best checkpoint           : VERIFIED")
print("Selection split           : Validation only")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Progress                  : 7 / 8 cells passed")
print("Next                      : package full run")

## Cell 8 — Package Teacher A seed-2026 full run

### Công việc

- verify best/last checkpoint và Validation artifacts;
- tạo manifest CSV/JSON/SHA256SUMS;
- tạo COMPLETE marker;
- ghi rõ Teacher quality gate đạt hay không;
- không tự động chuyển sang KD nếu quality gate chưa đạt.

### Thời gian dự kiến

1–5 phút, phụ thuộc kích thước checkpoint.

### Đầu ra bắt buộc

```text
Status                    : PASSED
Teacher A seed2026          : COMPLETE
Manifest verification     : ALL PASSED
Test Set status           : SEALED
Progress                  : 8 / 8 cells passed
```


In [ ]:
CELL_STARTED_AT = time.perf_counter()
training_audit = read_json(
    AUDIT_DIR / "phase3d_cell07_full_training.json"
)
assert training_audit["status"] == "passed"
assert LAST_CHECKPOINT.is_file()
assert BEST_CHECKPOINT.is_file()
validation_files = sorted(
    VALIDATION_DIR.glob("epoch_*_validation.json")
)
assert len(validation_files) == training_audit["completed_epochs"]

MANIFEST_CSV = MANIFEST_DIR / "teacher_a_seed2026_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "teacher_a_seed2026_manifest.json"
SHA256SUMS = MANIFEST_DIR / "teacher_a_seed2026_SHA256SUMS.txt"
COMPLETE_MARKER = OUTPUT_ROOT / "TEACHER_A_SEED2026_FULL_RUN_COMPLETE.json"
excluded = {
    MANIFEST_CSV.resolve(),
    MANIFEST_JSON.resolve(),
    SHA256SUMS.resolve(),
    COMPLETE_MARKER.resolve(),
}
rows = []
for artifact in sorted(
    OUTPUT_ROOT.rglob("*"),
    key=lambda value: value.as_posix(),
):
    if not artifact.is_file() or artifact.resolve() in excluded:
        continue
    if artifact.name.endswith(".tmp"):
        raise AssertionError(f"Temporary artifact: {artifact}")
    rows.append(
        {
            "relative_path": artifact.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "size_bytes": artifact.stat().st_size,
            "sha256": sha256_file(artifact),
        }
    )
with open(
    MANIFEST_CSV,
    "w",
    encoding="utf-8",
    newline="",
) as target:
    writer = csv.DictWriter(
        target,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase3_teacher_a_seed2026_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(rows),
        "total_size_bytes": sum(row["size_bytes"] for row in rows),
        "artifacts": rows,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
SHA256SUMS.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in rows
    ),
    encoding="utf-8",
)
with open(
    MANIFEST_CSV,
    "r",
    encoding="utf-8",
    newline="",
) as source:
    verified_rows = list(csv.DictReader(source))
for row in verified_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

complete_payload = {
    "marker_version": "phase3_teacher_a_seed2026_complete_v1",
    "created_at_utc": now_utc(),
    "status": "complete",
    "teacher_id": "A",
    "teacher": "CLIP ViT-B/32 + shared caption decoder",
    "seed": 2026,
    "completed_epochs": training_audit["completed_epochs"],
    "global_optimizer_steps": training_audit[
        "global_optimizer_steps"
    ],
    "best_result": training_audit["best_result"],
    "matched_student_seed": SEED,
    "matched_student_cider_d": MATCHED_STUDENT_CIDER_D,
    "quality_threshold": training_audit["quality_threshold"],
    "run_seed_contract": {
        "relative_path": str(
            RUN_SEED_CONTRACT_PATH.relative_to(OUTPUT_ROOT)
        ),
        "sha256": sha256_file(RUN_SEED_CONTRACT_PATH),
    },
    "quality_gate_passed": training_audit[
        "quality_gate_passed"
    ],
    "qualified_for_kd": training_audit["quality_gate_passed"],
    "best_checkpoint": {
        "relative_path": BEST_CHECKPOINT.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(BEST_CHECKPOINT),
    },
    "last_checkpoint": {
        "relative_path": LAST_CHECKPOINT.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(LAST_CHECKPOINT),
    },
    "manifests": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV),
        },
        "json": {
            "relative_path": MANIFEST_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_JSON),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SHA256SUMS),
        },
    },
    "authorized_next_action": (
        "After this run, wait for Teacher A seeds "
        "42 and 123, then aggregate all three "
        "runs on Validation. If the quality gate fails, retain this run "
        "as a documented auxiliary hypothesis."
    ),
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
}
write_json(COMPLETE_MARKER, complete_payload)

elapsed = time.perf_counter() - CELL_STARTED_AT
print("=" * 72)
print("PHASE 3D CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher A seed2026          : COMPLETE")
print("Completed epochs          :", training_audit["completed_epochs"])
print("Best Validation CIDEr-D   :", training_audit[
    "best_result"
]["metrics"]["CIDEr-D"])
print("Teacher quality gate      :", (
    "PASSED"
    if training_audit["quality_gate_passed"]
    else "NOT PASSED"
))
print("Qualified for KD          :", training_audit[
    "quality_gate_passed"
])
print("Validation artifacts      :", len(validation_files))
print("Manifest artifacts        :", len(rows))
print("Manifest verification     : ALL PASSED")
print("Best checkpoint           : VERIFIED")
print("Last checkpoint           : VERIFIED")
print("Selection split           : Validation only")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("COMPLETE marker           :", COMPLETE_MARKER)
print("COMPLETE SHA-256          :", sha256_file(COMPLETE_MARKER))
print("Progress                  : 8 / 8 cells passed")
print("Elapsed                   :", round(elapsed, 2), "giây")